# PatchTSMixer GPU Backtest on Colab

> **Deprecated:** Use `dl_runner.ipynb` instead. It supports both PatchTSMixer
> and AE+Ridge with MCP-driven cell-by-cell execution and Drive-persisted
> status tracking. Set `EXPERIMENT = "patchts"` in Cell 2.

Run a PatchTSMixer GPU backtest on Google Colab.

**Requirements:** Colab GPU runtime (T4 or better). Go to *Runtime > Change runtime type > GPU*.

In [ ]:
# --- 1. Setup: clone repo and install deps ---
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"  # FILL IN: your repo URL
REPO_DIR = "harxhar"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q torch transformers numpy pandas scikit-learn tqdm pyarrow

In [ ]:
# --- 2. Verify GPU and configure CUDA ---
from harxhar_dl.notebook_utils import configure_cuda, verify_gpu

verify_gpu()
configure_cuda()

## Data Configuration

In [ ]:
# --- 4. Upload your data ---
# Option A: Upload from local machine
# from google.colab import files
# uploaded = files.upload()  # upload your parquet files to all30min/

# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/harxhar_data/all30min'

# Option C: Data already in repo
DATA_PATH = "all30min"
RESULTS_DIR = "results_patchts"

## PatchTSMixer Backtest

In [ ]:
import copy

from harxhar_core.data import load_and_prep_data_strided
from harxhar_core.evaluation.metrics import calculate_global_metrics

from harxhar_dl.backtest.gpu_engine import run_multigpu_backtest
from harxhar_dl.config import DL_CONFIG

config = copy.deepcopy(DL_CONFIG)
config["data_path"] = DATA_PATH
config["gpu_count"] = 1  # Colab has 1 GPU

hparams = {
    "exog_cols": "none",
    "use_transform_exog": False,
    "use_diurnal": True,
    "allow_missing": False,
    "use_winsor": False,
}

X_np, y_np, dates, baselines, features = load_and_prep_data_strided(
    hparams, config["data_path"], lag=config["model"]["context_len"]
)
print(f"Data: {X_np.shape[0]:,} samples, {X_np.shape[1]} features")

In [ ]:
from harxhar_dl.notebook_utils import print_metrics

results = run_multigpu_backtest(
    X_np,
    y_np,
    dates,
    baselines,
    config,
    model_module="harxhar_dl.models.deep_learning",
)

print_metrics(calculate_global_metrics(results))

In [ ]:
from harxhar_dl.notebook_utils import save_results

csv_path = save_results(results, RESULTS_DIR, "patchts_results.csv")

In [ ]:
from harxhar_dl.notebook_utils import download_results

download_results(csv_path)